In [2]:
import pandas as pd

path = "/storage/vacoda/data/interim/hs_sync/target_h5/applied/comtrade_target_h5_hs_synced.parquet"

df = pd.read_parquet(path)

In [15]:
df.head(3)

,hs_sync_input_row_id,type_code,frequency_code,ref_period_id,year,month,period,reporter_code,reporter_iso,reporter_name,...,source_cmd_original,target_hs,target_cmd,allocation_weight,conversion_need,hs_sync_method,allocation_strategy,allocation_is_assumption,primary_value_original,primary_value_hs_synced
0,1,C,A,20170101,2017,52,2017,28,ATG,Antigua and Barbuda,...,391211,H5,391211,1.0,easy_without_weights,direct_1_to_1,deterministic,False,169.270,169.270
1,2,C,A,20170101,2017,52,2017,28,ATG,Antigua and Barbuda,...,391211,H5,391211,1.0,easy_without_weights,direct_1_to_1,deterministic,False,169.270,169.270
2,3,C,A,20170101,2017,52,2017,28,ATG,Antigua and Barbuda,...,391211,H5,391211,1.0,easy_without_weights,direct_1_to_1,deterministic,False,38.952,38.952


In [4]:
df.columns.tolist()

['hs_sync_input_row_id',
 'type_code',
 'frequency_code',
 'ref_period_id',
 'year',
 'month',
 'period',
 'reporter_code',
 'reporter_iso',
 'reporter_name',
 'flow_code',
 'flow_name',
 'partner_code',
 'partner_iso',
 'partner_name',
 'partner2_code',
 'partner2_iso',
 'partner2_name',
 'source_hs',
 'classification_search',
 'is_original_classification',
 'source_cmd',
 'commodity_name',
 'aggregation_level',
 'is_leaf',
 'customs_code',
 'customs_name',
 'mos_code',
 'mot_code',
 'mot_name',
 'quantity_unit_code',
 'quantity_unit',
 'quantity',
 'is_quantity_estimated',
 'alt_quantity_unit_code',
 'alt_quantity_unit',
 'alt_quantity',
 'is_alt_quantity_estimated',
 'net_weight_kg',
 'is_net_weight_estimated',
 'gross_weight_kg',
 'is_gross_weight_estimated',
 'cif_value_usd',
 'fob_value_usd',
 'primary_value',
 'legacy_estimation_flag',
 'is_reported',
 'is_aggregate',
 'source_hs_original',
 'source_cmd_original',
 'target_hs',
 'target_cmd',
 'allocation_weight',
 'conversion_n

In [6]:
value_columns =['cif_value_usd',
 'fob_value_usd',
 'primary_value']

df[value_columns].describe()

,cif_value_usd,fob_value_usd,primary_value
count,3.345576e+07,3.344003e+07,4.991792e+07
mean,8.056602e+05,1.088909e+06,1.187314e+06
std,1.709283e+07,2.401075e+07,2.327669e+07
min,0.000000e+00,0.000000e+00,0.000000e+00
25%,7.119700e+01,1.159340e+02,4.822450e+02
50%,1.992000e+03,3.340467e+03,6.148100e+03
75%,3.630400e+04,5.159355e+04,7.244236e+04
max,1.139287e+10,1.298835e+10,1.298835e+10


In [7]:
value_cols = [
    "primary_value",
    "cif_value_usd",
    "fob_value_usd",
]

df[value_cols].isna().value_counts().sort_index()

primary_value  cif_value_usd  fob_value_usd
False          False          False            16977876
                              True             16477885
               True           False            16462156
Name: count, dtype: int64

In [8]:
pd.DataFrame({
    "rows": [
        len(df),
        df["primary_value"].isna().sum(),
        (
            df["primary_value"].isna()
            & df["cif_value_usd"].notna()
        ).sum(),
        (
            df["primary_value"].isna()
            & df["fob_value_usd"].notna()
        ).sum(),
        (
            df["primary_value"].isna()
            & (
                df["cif_value_usd"].notna()
                | df["fob_value_usd"].notna()
            )
        ).sum(),
    ]
}, index=[
    "total_rows",
    "primary_missing",
    "primary_missing_but_cif_present",
    "primary_missing_but_fob_present",
    "primary_missing_but_any_alternative_present",
])

,rows
total_rows,49917917
primary_missing,0
primary_missing_but_cif_present,0
primary_missing_but_fob_present,0
primary_missing_but_any_alternative_present,0


In [9]:
df[value_cols].eq(0).sum()

primary_value         44
cif_value_usd    5202168
fob_value_usd    4077697
dtype: int64

In [10]:
availability = (
    df[
        [
            "primary_value",
            "cif_value_usd",
            "fob_value_usd",
        ]
    ]
    .notna()
    .value_counts()
    .rename("rows")
    .reset_index()
)

availability

,primary_value,cif_value_usd,fob_value_usd,rows
0,True,True,True,16977876
1,True,True,False,16477885
2,True,False,True,16462156


In [11]:
comparison = pd.DataFrame({
    "primary_equals_cif": (
        df["primary_value"].eq(df["cif_value_usd"])
        & df["cif_value_usd"].notna()
    ),
    "primary_equals_fob": (
        df["primary_value"].eq(df["fob_value_usd"])
        & df["fob_value_usd"].notna()
    ),
})

comparison.value_counts().rename("rows")

primary_equals_cif  primary_equals_fob
True                False                 26510456
False               True                  21732470
True                True                   1674991
Name: rows, dtype: int64

In [12]:
pd.Series({
    "primary equals CIF": (
        df["primary_value"].eq(df["cif_value_usd"])
        & df["cif_value_usd"].notna()
    ).sum(),

    "primary equals FOB": (
        df["primary_value"].eq(df["fob_value_usd"])
        & df["fob_value_usd"].notna()
    ).sum(),

    "primary equals both": (
        df["primary_value"].eq(df["cif_value_usd"])
        & df["primary_value"].eq(df["fob_value_usd"])
        & df["cif_value_usd"].notna()
        & df["fob_value_usd"].notna()
    ).sum(),

    "primary differs from both": (
        df["cif_value_usd"].notna()
        & df["fob_value_usd"].notna()
        & df["primary_value"].ne(df["cif_value_usd"])
        & df["primary_value"].ne(df["fob_value_usd"])
    ).sum(),
})


primary equals CIF           28185447
primary equals FOB           23407461
primary equals both           1674991
primary differs from both           0
dtype: int64

In [13]:
df.assign(
    primary_equals_cif=(
        df["primary_value"].eq(df["cif_value_usd"])
        & df["cif_value_usd"].notna()
    ),
    primary_equals_fob=(
        df["primary_value"].eq(df["fob_value_usd"])
        & df["fob_value_usd"].notna()
    ),
).groupby("flow_code").agg(
    rows=("primary_value", "size"),
    cif_available=("cif_value_usd", "count"),
    fob_available=("fob_value_usd", "count"),
    primary_equals_cif=("primary_equals_cif", "sum"),
    primary_equals_fob=("primary_equals_fob", "sum"),
)

,rows,cif_available,fob_available,primary_equals_cif,primary_equals_fob
flow_code,,,,,
DX,4485514,1100496,4485514,3478,4485514
FM,4465801,4275553,1815395,4095596,563843
M,24938179,23914484,11396053,23573211,2766223
MIP,126993,126993,71503,126993,6108
MOP,28759,28759,8960,28759,440
RM,373089,341063,163033,333977,85759
RX,409186,156055,409186,3924,409186
X,14952413,3442777,14952405,19297,14952405
XIP,34433,9468,34433,0,34433


In [16]:
keys = [
    "period",
    "reporter_code",
    "partner_code",
    "target_cmd",
]

export_check = (
    df[df["flow_code"].isin(["X", "DX", "RX"])]
    .groupby(keys + ["flow_code"], as_index=False)["primary_value"]
    .sum()
    .pivot_table(
        index=keys,
        columns="flow_code",
        values="primary_value",
        fill_value=0,
    )
    .reset_index()
)

export_check["dx_plus_rx"] = (
    export_check.get("DX", 0)
    + export_check.get("RX", 0)
)

export_check["difference"] = (
    export_check.get("X", 0)
    - export_check["dx_plus_rx"]
)

export_check["relative_difference"] = (
    export_check["difference"].abs()
    / export_check.get("X", 0).replace(0, pd.NA)
)

export_check["relative_difference"].describe(
    percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]
)

count     3867229.0
unique      49075.0
top             1.0
freq      2859467.0
Name: relative_difference, dtype: float64

In [17]:
pd.Series({
    "comparable_groups": export_check["relative_difference"].notna().sum(),
    "within_1_percent": export_check["relative_difference"].le(0.01).sum(),
    "within_5_percent": export_check["relative_difference"].le(0.05).sum(),
    "within_10_percent": export_check["relative_difference"].le(0.10).sum(),
})

comparable_groups    3867229
within_1_percent     1002841
within_5_percent     1003356
within_10_percent    1003671
dtype: int64

In [18]:
import numpy as np

x_value = export_check.get("X", 0)

export_check["relative_difference"] = np.where(
    x_value > 0,
    export_check["difference"].abs() / x_value,
    np.nan,
)

export_check["relative_difference"].describe(
    percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]
)

count    3.867229e+06
mean     7.401726e-01
std      4.383936e-01
min      0.000000e+00
50%      1.000000e+00
75%      1.000000e+00
90%      1.000000e+00
95%      1.000000e+00
99%      1.000000e+00
max      1.000000e+00
Name: relative_difference, dtype: float64

In [19]:

comparable = export_check.loc[
    export_check.get("X", 0).gt(0)
    & (
        export_check.get("DX", 0).gt(0)
        | export_check.get("RX", 0).gt(0)
    )
].copy()

comparable["dx_plus_rx"] = (
    comparable.get("DX", 0)
    + comparable.get("RX", 0)
)

comparable["relative_difference"] = (
    (
        comparable["X"]
        - comparable["dx_plus_rx"]
    ).abs()
    / comparable["X"]
)

pd.Series({
    "comparable_groups": len(comparable),
    "within_1_percent": comparable["relative_difference"].le(0.01).sum(),
    "within_5_percent": comparable["relative_difference"].le(0.05).sum(),
    "within_10_percent": comparable["relative_difference"].le(0.10).sum(),
})

comparable_groups    1007762
within_1_percent     1002841
within_5_percent     1003356
within_10_percent    1003671
dtype: int64

In [20]:
pd.Series({
    "within_1_percent_pct":
        comparable["relative_difference"].le(0.01).mean() * 100,

    "within_5_percent_pct":
        comparable["relative_difference"].le(0.05).mean() * 100,

    "within_10_percent_pct":
        comparable["relative_difference"].le(0.10).mean() * 100,
})

within_1_percent_pct     99.511690
within_5_percent_pct     99.562794
within_10_percent_pct    99.594051
dtype: float64

In [21]:
export_check["has_x"] = export_check.get("X", 0).gt(0)
export_check["has_dx"] = export_check.get("DX", 0).gt(0)
export_check["has_rx"] = export_check.get("RX", 0).gt(0)

export_check["has_detailed_export"] = (
    export_check["has_dx"]
    | export_check["has_rx"]
)

pd.Series({
    "groups_with_x": export_check["has_x"].sum(),

    "groups_with_detailed_export":
        export_check["has_detailed_export"].sum(),

    "detailed_groups_without_x": (
        ~export_check["has_x"]
        & export_check["has_detailed_export"]
    ).sum(),

    "dx_without_x": (
        ~export_check["has_x"]
        & export_check["has_dx"]
    ).sum(),

    "rx_without_x": (
        ~export_check["has_x"]
        & export_check["has_rx"]
    ).sum(),
})

groups_with_x                  3867229
groups_with_detailed_export    1007762
detailed_groups_without_x            0
dx_without_x                         0
rx_without_x                         0
dtype: int64

In [22]:
export_fallback = export_check.loc[
    ~export_check["has_x"]
    & export_check["has_detailed_export"]
].copy()

pd.Series({
    "fallback_dx_value":
        export_fallback.get("DX", 0).sum(),

    "fallback_rx_value":
        export_fallback.get("RX", 0).sum(),

    "fallback_total_export_value":
        (
            export_fallback.get("DX", 0)
            + export_fallback.get("RX", 0)
        ).sum(),
})

fallback_dx_value              0.0
fallback_rx_value              0.0
fallback_total_export_value    0.0
dtype: float64

In [24]:
## 
value_columns = ['primary_value', 'primary_value_hs_synced', 'primary_value_original']
df[value_columns].describe()

,primary_value,primary_value_hs_synced,primary_value_original
count,4.991792e+07,4.991792e+07,4.991792e+07
mean,1.187314e+06,1.163947e+06,1.187314e+06
std,2.327669e+07,2.191921e+07,2.327669e+07
min,0.000000e+00,0.000000e+00,0.000000e+00
25%,4.822450e+02,4.736500e+02,4.822450e+02
50%,6.148100e+03,6.033479e+03,6.148100e+03
75%,7.244236e+04,7.107164e+04,7.244236e+04
max,1.298835e+10,1.298835e+10,1.298835e+10


In [25]:
import duckdb

path = (
    "/storage/vacoda/data/interim/cif_fob/"
    "target_h5/target_h5_cif_fob_mirror_pairs.parquet"
)

df = duckdb.sql(
    f"""
    SELECT *
    FROM read_parquet('{path}')
    LIMIT 100
    """
).df()

df.head()

,year,exporter_iso,importer_iso,target_cmd,export_value_fob,import_value_cif,export_net_weight_kg,import_net_weight_kg,export_direct_fob_value,export_direct_cif_value,...,has_export_report,has_import_report,has_both_mirrors,raw_value_ratio,export_unit_value_usd_per_kg,import_unit_value_usd_per_kg,unit_value_ratio,quantity_concordance,exclusion_reason,training_eligible
0,2023,USA,NLD,440399,1.237298e+06,4.291339e+04,4.608168e+06,27263.0,1.237298e+06,NaN,...,True,True,True,0.034683,0.268501,1.574052,5.862370,0.005916,low_quantity_concordance,False
1,2023,USA,THA,440399,6.409000e+04,6.794644e+03,2.386955e+05,8834.0,6.409000e+04,NaN,...,True,True,True,0.106017,0.268501,0.769147,2.864596,0.037009,low_quantity_concordance,False
2,2024,BEL,PRT,440399,2.460930e+02,4.825988e+04,7.352000e+01,69513.0,2.460930e+02,NaN,...,True,True,True,196.104249,3.347293,0.694257,0.207408,0.001058,low_quantity_concordance,False
3,2024,CAN,CHN,440399,1.065437e+07,1.403722e+07,4.277231e+07,22764763.0,1.065437e+07,NaN,...,True,True,True,1.317507,0.249095,0.616620,2.475441,0.532231,low_quantity_concordance,False
4,2024,HRV,BIH,440399,8.581400e+04,1.365331e+05,4.841800e+05,1090633.0,8.581400e+04,NaN,...,True,True,True,1.591035,0.177236,0.125187,0.706330,0.443944,low_quantity_concordance,False


In [26]:
duckdb.sql(
    f"""
    SELECT
        exclusion_reason,
        COUNT(*) AS n_rows,
        ROUND(
            100.0 * COUNT(*) / SUM(COUNT(*)) OVER (),
            2
        ) AS share_percent
    FROM read_parquet('{path}')
    GROUP BY exclusion_reason
    ORDER BY n_rows DESC
    """
).df()

,exclusion_reason,n_rows,share_percent
0,low_quantity_concordance,1972869,35.95
1,missing_export_report,1831489,33.37
2,missing_import_report,1328738,24.21
3,eligible,303681,5.53
4,missing_or_nonpositive_import_weight,29993,0.55
5,missing_or_nonpositive_export_weight,21210,0.39
6,nonpositive_export_value,1,0.00


In [27]:
duckdb.sql(
    f"""
    SELECT
        QUANTILE_CONT(
            quantity_concordance,
            [0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
        ) AS concordance_quantiles
    FROM read_parquet('{path}')
    WHERE quantity_concordance IS NOT NULL
    """
).df()

,concordance_quantiles
0,"[0.0001696709527056308, 0.0034566062186056684,..."


In [29]:
duckdb.sql(
    f"""
    SELECT
        COUNT(*) AS n_eligible,
        MEDIAN(raw_value_ratio) AS median_value_ratio,
        MEDIAN(unit_value_ratio) AS median_unit_value_ratio,
        QUANTILE_CONT(
            unit_value_ratio,
            [0.01, 0.05, 0.50, 0.95, 0.99]
        ) AS unit_value_ratio_quantiles
    FROM read_parquet('{path}')
    WHERE training_eligible
    """
).df()

,n_eligible,median_value_ratio,median_unit_value_ratio,unit_value_ratio_quantiles
0,303681,1.047962,1.048389,"[0.1622232548794118, 0.5279949415938775, 1.048..."


In [30]:
import duckdb

path = (
    "/storage/vacoda/data/interim/cif_fob/"
    "target_h5/target_h5_cif_fob_mirror_pairs.parquet"
)

duckdb.sql(
    f"""
    SELECT
        COUNT(*) AS n_eligible,

        COUNT(*) FILTER (
            WHERE unit_value_ratio < 1
        ) AS n_ratio_below_1,

        COUNT(*) FILTER (
            WHERE unit_value_ratio BETWEEN 1 AND 1.5
        ) AS n_ratio_1_to_1_5,

        COUNT(*) FILTER (
            WHERE unit_value_ratio > 1.5
        ) AS n_ratio_above_1_5,

        ROUND(
            100.0 * COUNT(*) FILTER (
                WHERE unit_value_ratio < 1
            ) / COUNT(*),
            2
        ) AS share_below_1_percent

    FROM read_parquet('{path}')
    WHERE training_eligible
    """
).df()

,n_eligible,n_ratio_below_1,n_ratio_1_to_1_5,n_ratio_above_1_5,share_below_1_percent
0,303681,97331,174653,31697,32.05


In [33]:
import duckdb

path_hs_sync = (
    "/storage/vacoda/data/interim/hs_sync/target_h5/applied/"
    "comtrade_target_h5_hs_synced.parquet"
)

duckdb.sql(
    f"""
    SELECT
        is_quantity_estimated,
        is_net_weight_estimated,
        COUNT(*) AS n_rows,

        MEDIAN(quantity / net_weight_kg) FILTER (
            WHERE quantity > 0
              AND net_weight_kg > 0
              AND LOWER(quantity_unit) = 'kg'
        ) AS median_quantity_to_net_weight

    FROM read_parquet('{path_hs_sync}')

    GROUP BY
        is_quantity_estimated,
        is_net_weight_estimated

    ORDER BY
        is_quantity_estimated,
        is_net_weight_estimated
    """
).df()

,is_quantity_estimated,is_net_weight_estimated,n_rows,median_quantity_to_net_weight
0,False,False,41037308,1.000000
1,False,True,557736,0.999173
2,True,False,2800718,1.161457
3,True,True,5522155,1.000000


In [34]:
duckdb.sql(
    f"""
    SELECT
        is_quantity_estimated,
        is_net_weight_estimated,

        COUNT(*) FILTER (
            WHERE quantity > 0
              AND net_weight_kg > 0
              AND LOWER(TRIM(quantity_unit)) = 'kg'
        ) AS n_comparable_kg_rows,

        MEDIAN(quantity / net_weight_kg) FILTER (
            WHERE quantity > 0
              AND net_weight_kg > 0
              AND LOWER(TRIM(quantity_unit)) = 'kg'
        ) AS median_quantity_to_net_weight,

        QUANTILE_CONT(
            quantity / net_weight_kg,
            [0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
        ) FILTER (
            WHERE quantity > 0
              AND net_weight_kg > 0
              AND LOWER(TRIM(quantity_unit)) = 'kg'
        ) AS ratio_quantiles

    FROM read_parquet('{path_hs_sync}')

    GROUP BY
        is_quantity_estimated,
        is_net_weight_estimated

    ORDER BY
        is_quantity_estimated,
        is_net_weight_estimated
    """
).df()

,is_quantity_estimated,is_net_weight_estimated,n_comparable_kg_rows,median_quantity_to_net_weight,ratio_quantiles
0,False,False,37547512,1.000000,"[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]"
1,False,True,2,0.999173,"[0.9991730843553875, 0.9991730843553874, 0.999..."
2,True,False,570,1.161457,"[0.011277777777777779, 0.3178672421601963, 0.7..."
3,True,True,5035355,1.000000,"[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]"


In [35]:
import duckdb

path_hs_sync = (
    "/storage/vacoda/data/interim/hs_sync/target_h5/applied/"
    "comtrade_target_h5_hs_synced.parquet"
)

duckdb.sql(
    f"""
    WITH classified AS (
        SELECT
            *,

            net_weight_kg > 0
                AS has_net_weight,

            quantity > 0
            AND LOWER(TRIM(quantity_unit)) = 'kg'
                AS has_quantity_kg,

            alt_quantity > 0
            AND LOWER(TRIM(alt_quantity_unit)) = 'kg'
                AS has_alt_quantity_kg

        FROM read_parquet('{path_hs_sync}')
        WHERE flow_code IN ('M', 'X')
    )

    SELECT
        COUNT(*) AS total_rows,

        COUNT(*) FILTER (
            WHERE has_net_weight
        ) AS rows_with_net_weight,

        COUNT(*) FILTER (
            WHERE NOT has_net_weight
              AND has_quantity_kg
        ) AS recovered_by_quantity_kg,

        COUNT(*) FILTER (
            WHERE NOT has_net_weight
              AND NOT has_quantity_kg
              AND has_alt_quantity_kg
        ) AS recovered_by_alt_quantity_kg,

        COUNT(*) FILTER (
            WHERE NOT has_net_weight
              AND NOT has_quantity_kg
              AND NOT has_alt_quantity_kg
        ) AS still_without_kg_measure,

        ROUND(
            100.0 * COUNT(*) FILTER (
                WHERE NOT has_net_weight
                  AND NOT has_quantity_kg
                  AND has_alt_quantity_kg
            ) / COUNT(*),
            4
        ) AS alt_quantity_recovery_share_percent

    FROM classified
    """
).df()

,total_rows,rows_with_net_weight,recovered_by_quantity_kg,recovered_by_alt_quantity_kg,still_without_kg_measure,alt_quantity_recovery_share_percent
0,39890592,39437037,61075,19315,53234,0.0484


In [36]:
duckdb.sql(
    f"""
    WITH classified AS (
        SELECT
            *,

            net_weight_kg > 0
                AS has_net_weight,

            quantity > 0
            AND LOWER(TRIM(quantity_unit)) = 'kg'
                AS has_quantity_kg,

            alt_quantity > 0
            AND LOWER(TRIM(alt_quantity_unit)) = 'kg'
                AS has_alt_quantity_kg,

            gross_weight_kg > 0
                AS has_gross_weight

        FROM read_parquet('{path_hs_sync}')
        WHERE flow_code IN ('M', 'X')
    )

    SELECT
        COUNT(*) AS total_rows,

        COUNT(*) FILTER (
            WHERE has_net_weight
        ) AS rows_with_net_weight,

        COUNT(*) FILTER (
            WHERE NOT has_net_weight
              AND has_quantity_kg
        ) AS recovered_by_quantity_kg,

        COUNT(*) FILTER (
            WHERE NOT has_net_weight
              AND NOT has_quantity_kg
              AND has_alt_quantity_kg
        ) AS recovered_by_alt_quantity_kg,

        COUNT(*) FILTER (
            WHERE NOT has_net_weight
              AND NOT has_quantity_kg
              AND NOT has_alt_quantity_kg
              AND has_gross_weight
        ) AS recovered_by_gross_weight,

        COUNT(*) FILTER (
            WHERE NOT has_net_weight
              AND NOT has_quantity_kg
              AND NOT has_alt_quantity_kg
              AND NOT has_gross_weight
        ) AS still_without_any_kg_measure,

        ROUND(
            100.0 * COUNT(*) FILTER (
                WHERE NOT has_net_weight
                  AND NOT has_quantity_kg
                  AND NOT has_alt_quantity_kg
                  AND has_gross_weight
            ) / COUNT(*),
            4
        ) AS gross_weight_recovery_share_percent

    FROM classified
    """
).df()

,total_rows,rows_with_net_weight,recovered_by_quantity_kg,recovered_by_alt_quantity_kg,recovered_by_gross_weight,still_without_any_kg_measure,gross_weight_recovery_share_percent
0,39890592,39437037,61075,19315,1646,51580,0.0041


In [37]:
duckdb.sql(
    f"""
    WITH candidates AS (
        SELECT
            gross_weight_kg,
            is_gross_weight_estimated
        FROM read_parquet('{path_hs_sync}')
        WHERE flow_code IN ('M', 'X')

          AND (
              net_weight_kg IS NULL
              OR net_weight_kg <= 0
          )

          AND NOT (
              quantity > 0
              AND LOWER(TRIM(quantity_unit)) = 'kg'
          )

          AND NOT (
              alt_quantity > 0
              AND LOWER(TRIM(alt_quantity_unit)) = 'kg'
          )

          AND gross_weight_kg > 0
    )

    SELECT
        COALESCE(is_gross_weight_estimated, FALSE)
            AS is_gross_weight_estimated,
        COUNT(*) AS n_rows,
        SUM(gross_weight_kg) AS total_gross_weight_kg

    FROM candidates

    GROUP BY
        COALESCE(is_gross_weight_estimated, FALSE)

    ORDER BY
        is_gross_weight_estimated
    """
).df()

,is_gross_weight_estimated,n_rows,total_gross_weight_kg
0,False,17221,3.827856e+10


In [38]:
duckdb.sql(
    f"""
    WITH classified AS (
        SELECT
            COALESCE(net_weight_kg > 0, FALSE)
                AS has_net_weight,

            COALESCE(
                quantity > 0
                AND LOWER(TRIM(quantity_unit)) = 'kg',
                FALSE
            ) AS has_quantity_kg,

            COALESCE(
                alt_quantity > 0
                AND LOWER(TRIM(alt_quantity_unit)) = 'kg',
                FALSE
            ) AS has_alt_quantity_kg,

            COALESCE(gross_weight_kg > 0, FALSE)
                AS has_gross_weight

        FROM read_parquet('{path_hs_sync}')
        WHERE flow_code IN ('M', 'X')
    )

    SELECT
        COUNT(*) AS total_rows,

        COUNT(*) FILTER (
            WHERE has_net_weight
        ) AS rows_with_net_weight,

        COUNT(*) FILTER (
            WHERE NOT has_net_weight
              AND has_quantity_kg
        ) AS recovered_by_quantity_kg,

        COUNT(*) FILTER (
            WHERE NOT has_net_weight
              AND NOT has_quantity_kg
              AND has_alt_quantity_kg
        ) AS recovered_by_alt_quantity_kg,

        COUNT(*) FILTER (
            WHERE NOT has_net_weight
              AND NOT has_quantity_kg
              AND NOT has_alt_quantity_kg
              AND has_gross_weight
        ) AS recovered_by_gross_weight,

        COUNT(*) FILTER (
            WHERE NOT has_net_weight
              AND NOT has_quantity_kg
              AND NOT has_alt_quantity_kg
              AND NOT has_gross_weight
        ) AS still_without_any_kg_measure

    FROM classified
    """
).df()

,total_rows,rows_with_net_weight,recovered_by_quantity_kg,recovered_by_alt_quantity_kg,recovered_by_gross_weight,still_without_any_kg_measure
0,39890592,39437037,121587,118203,17221,196544


In [1]:
import duckdb

training_path = (
    "/storage/vacoda/data/interim/cif_fob/target_h5/"
    "target_h5_cif_fob_training_sample.parquet"
)

duckdb.sql(
    f"""
    SELECT
        factor_source,
        COUNT(*) AS n_rows,

        MEDIAN(observed_cif_fob_factor) AS median_factor,

        QUANTILE_CONT(
            observed_cif_fob_factor,
            0.01
        ) AS p01,

        QUANTILE_CONT(
            observed_cif_fob_factor,
            0.05
        ) AS p05,

        QUANTILE_CONT(
            observed_cif_fob_factor,
            0.95
        ) AS p95,

        QUANTILE_CONT(
            observed_cif_fob_factor,
            0.99
        ) AS p99,

        ROUND(
            100.0 * COUNT(*) FILTER (
                WHERE observed_cif_fob_factor < 1
            ) / COUNT(*),
            2
        ) AS share_below_1_percent,

        ROUND(
            100.0 * COUNT(*) FILTER (
                WHERE observed_cif_fob_factor > 1.5
            ) / COUNT(*),
            2
        ) AS share_above_1_5_percent

    FROM read_parquet('{training_path}')

    WHERE model_eligible

    GROUP BY factor_source
    ORDER BY factor_source
    """
).df()

,factor_source,n_rows,median_factor,p01,p05,p95,p99,share_below_1_percent,share_above_1_5_percent
0,direct_import_cif_fob,683328,1.074113,1.000000,1.000000,1.735534,6.77770,0.57,7.14
1,mirror_unit_value,252947,1.044246,0.155277,0.516059,2.133497,7.46041,33.26,10.52


In [2]:
duckdb.sql(
    f"""
    SELECT
        COUNT(*) AS n_with_both_factors,

        MEDIAN(direct_cif_fob_factor)
            AS median_direct_factor,

        MEDIAN(mirror_unit_value_factor)
            AS median_mirror_factor,

        MEDIAN(
            direct_cif_fob_factor
            / mirror_unit_value_factor
        ) AS median_direct_to_mirror_ratio,

        QUANTILE_CONT(
            direct_cif_fob_factor
            / mirror_unit_value_factor,
            [0.05, 0.25, 0.50, 0.75, 0.95]
        ) AS comparison_quantiles

    FROM read_parquet('{training_path}')

    WHERE direct_cif_fob_factor > 0
      AND mirror_unit_value_factor > 0
      AND high_weight_concordance
    """
).df()

,n_with_both_factors,median_direct_factor,median_mirror_factor,median_direct_to_mirror_ratio,comparison_quantiles
0,51899,1.067795,1.067709,1.000222,"[0.5697458407026997, 0.9343470245644956, 1.000..."


In [3]:
duckdb.sql(
    f"""
    SELECT
        target_cmd,
        COUNT(*) AS n_rows,
        MEDIAN(direct_cif_fob_factor) AS median_factor,
        QUANTILE_CONT(direct_cif_fob_factor, 0.95) AS p95,
        QUANTILE_CONT(direct_cif_fob_factor, 0.99) AS p99
    FROM read_parquet('{training_path}')
    WHERE direct_cif_fob_factor > 0
    GROUP BY target_cmd
    HAVING COUNT(*) >= 100
    ORDER BY p99 DESC
    LIMIT 50
    """
).df()

,target_cmd,n_rows,median_factor,p95,p99
0,470329,835,1.074114,2.077611,56.402300
1,480210,1688,1.074156,2.817175,35.914677
2,470500,317,1.081913,1.889326,35.849162
3,470691,432,1.072987,1.571499,31.554233
4,470200,426,1.061625,1.431165,28.367001
5,482040,1946,1.101184,4.093559,27.790923
6,440121,776,1.066825,2.186657,25.269549
7,470620,708,1.094109,1.924714,24.282700
8,470100,556,1.090590,1.765897,23.791935
9,481099,2975,1.081860,2.420768,22.191711


## Inspect Mirror-Factor Cases

In [ ]:

import duckdb
import pandas as pd

training_path = (
    "/storage/vacoda/data/interim/cif_fob/target_h5/"
    "target_h5_cif_fob_training_sample.parquet"
)

mirror_examples = duckdb.sql(
    f"""
    SELECT
        year,
        exporter_iso,
        importer_iso,
        target_cmd,

        export_value_fob,
        import_value_cif,

        export_weight_kg,
        import_weight_kg,
        weight_concordance,

        export_unit_value_usd_per_kg,
        import_unit_value_usd_per_kg,

        raw_value_ratio,
        mirror_unit_value_factor,

        export_source_rows,
        import_source_rows,

        export_proxy_weight_rows,
        import_proxy_weight_rows,
        export_estimated_weight_rows,
        import_estimated_weight_rows

    FROM read_parquet('{training_path}')

    WHERE factor_source = 'mirror_unit_value'

    ORDER BY RANDOM()
    LIMIT 20
    """
).df()

mirror_examples

,year,exporter_iso,importer_iso,target_cmd,export_value_fob,import_value_cif,export_weight_kg,import_weight_kg,weight_concordance,export_unit_value_usd_per_kg,import_unit_value_usd_per_kg,raw_value_ratio,mirror_unit_value_factor,export_source_rows,import_source_rows,export_proxy_weight_rows,import_proxy_weight_rows,export_estimated_weight_rows,import_estimated_weight_rows
0,2023,CHN,GBR,491191,1.322924e+07,1.584340e+07,2.848463e+06,2.962048e+06,0.961653,4.644345,5.348801,1.197605,1.151680,1,31,0,0,0,1
1,2024,POL,BIH,480258,4.821900e+04,5.309173e+04,4.150000e+04,4.210356e+04,0.985665,1.161904,1.260980,1.101054,1.085270,1,4,0,0,0,0
2,2018,RUS,AZE,480257,1.332932e+06,1.338436e+06,1.461943e+06,1.446110e+06,0.989170,0.911754,0.925543,1.004129,1.015123,1,1,0,0,0,0
3,2019,TUR,HUN,940161,7.676310e+05,9.132710e+05,1.060120e+05,1.091920e+05,0.970877,7.240982,8.363900,1.189727,1.155078,2,1,0,0,0,0
4,2019,CHE,CRI,491000,4.678920e+02,7.371400e+02,3.900000e+01,4.090100e+01,0.953522,11.997231,18.022542,1.575449,1.502225,2,1,0,0,0,1
5,2019,BRA,IND,480100,8.545000e+04,1.052763e+05,2.034530e+05,2.030000e+05,0.997773,0.419999,0.518603,1.232023,1.234772,1,1,0,0,0,0
6,2022,SWE,SGP,480100,2.744840e+06,3.116558e+06,3.680000e+06,3.815650e+06,0.964449,0.745880,0.816783,1.135424,1.095059,1,1,0,0,0,0
7,2018,LBN,AUT,481910,4.200000e+01,1.709029e+03,2.504700e+01,2.500000e+01,0.998124,1.676848,68.361160,40.691167,40.767666,1,1,0,0,1,0
8,2019,LUX,DEU,481013,1.870544e+05,1.881825e+05,9.135319e+04,8.883400e+04,0.972424,2.047596,2.118362,1.006031,1.034560,2,2,0,0,0,0
9,2021,DEU,MMR,481200,2.055515e+04,2.351414e+04,1.836000e+03,1.971625e+03,0.931212,11.195614,11.926274,1.143954,1.065263,2,2,0,0,0,0


In [5]:
mirror_examples = mirror_examples.assign(
    calculated_export_unit_value=(
        mirror_examples["export_value_fob"]
        / mirror_examples["export_weight_kg"]
    ),
    calculated_import_unit_value=(
        mirror_examples["import_value_cif"]
        / mirror_examples["import_weight_kg"]
    ),
)

mirror_examples["calculated_mirror_factor"] = (
    mirror_examples["calculated_import_unit_value"]
    / mirror_examples["calculated_export_unit_value"]
)

mirror_examples[
    [
        "year",
        "exporter_iso",
        "importer_iso",
        "target_cmd",
        "export_value_fob",
        "import_value_cif",
        "export_weight_kg",
        "import_weight_kg",
        "weight_concordance",
        "calculated_export_unit_value",
        "calculated_import_unit_value",
        "calculated_mirror_factor",
        "mirror_unit_value_factor",
    ]
]

,year,exporter_iso,importer_iso,target_cmd,export_value_fob,import_value_cif,export_weight_kg,import_weight_kg,weight_concordance,calculated_export_unit_value,calculated_import_unit_value,calculated_mirror_factor,mirror_unit_value_factor
0,2023,CHN,GBR,491191,1.322924e+07,1.584340e+07,2.848463e+06,2.962048e+06,0.961653,4.644345,5.348801,1.151680,1.151680
1,2024,POL,BIH,480258,4.821900e+04,5.309173e+04,4.150000e+04,4.210356e+04,0.985665,1.161904,1.260980,1.085270,1.085270
2,2018,RUS,AZE,480257,1.332932e+06,1.338436e+06,1.461943e+06,1.446110e+06,0.989170,0.911754,0.925543,1.015123,1.015123
3,2019,TUR,HUN,940161,7.676310e+05,9.132710e+05,1.060120e+05,1.091920e+05,0.970877,7.240982,8.363900,1.155078,1.155078
4,2019,CHE,CRI,491000,4.678920e+02,7.371400e+02,3.900000e+01,4.090100e+01,0.953522,11.997231,18.022542,1.502225,1.502225
5,2019,BRA,IND,480100,8.545000e+04,1.052763e+05,2.034530e+05,2.030000e+05,0.997773,0.419999,0.518603,1.234772,1.234772
6,2022,SWE,SGP,480100,2.744840e+06,3.116558e+06,3.680000e+06,3.815650e+06,0.964449,0.745880,0.816783,1.095059,1.095059
7,2018,LBN,AUT,481910,4.200000e+01,1.709029e+03,2.504700e+01,2.500000e+01,0.998124,1.676848,68.361160,40.767666,40.767666
8,2019,LUX,DEU,481013,1.870544e+05,1.881825e+05,9.135319e+04,8.883400e+04,0.972424,2.047596,2.118362,1.034560,1.034560
9,2021,DEU,MMR,481200,2.055515e+04,2.351414e+04,1.836000e+03,1.971625e+03,0.931212,11.195614,11.926274,1.065263,1.065263


## Inspect Direct-Factor Cases

In [6]:
near_threshold = duckdb.sql(
    f"""
    SELECT
        year,
        exporter_iso,
        importer_iso,
        target_cmd,

        export_value_fob,
        import_value_cif,

        export_weight_kg,
        import_weight_kg,
        weight_concordance,

        export_unit_value_usd_per_kg,
        import_unit_value_usd_per_kg,
        mirror_unit_value_factor

    FROM read_parquet('{training_path}')

    WHERE unit_value_pair_eligible
      AND weight_concordance BETWEEN 0.895 AND 0.905

    ORDER BY weight_concordance
    LIMIT 30
    """
).df()

near_threshold

,year,exporter_iso,importer_iso,target_cmd,export_value_fob,import_value_cif,export_weight_kg,import_weight_kg,weight_concordance,export_unit_value_usd_per_kg,import_unit_value_usd_per_kg,mirror_unit_value_factor
0,2020,NLD,HRV,470421,4.569000e+00,6.000000e+00,2.000000e+00,1.790000e+00,0.895000,2.284500,3.351955,1.467260
1,2018,CAN,IRL,490890,6.484300e+01,2.102100e+02,8.950000e-01,1.000000e+00,0.895000,72.450279,210.210000,2.901438
2,2022,GRC,CYP,480591,2.369360e+02,3.580370e+02,2.000000e+02,1.790000e+02,0.895000,1.184680,2.000207,1.688394
3,2021,SAU,ARE,482361,9.645330e+02,6.695710e+02,3.580000e+02,4.000000e+02,0.895000,2.694226,1.673928,0.621302
4,2020,ITA,IRN,391239,8.374581e+03,4.727000e+03,2.000000e+02,1.790000e+02,0.895000,41.872905,26.407821,0.630666
5,2019,HUN,GRC,490890,1.140000e+02,1.119000e+00,8.950000e-01,1.000000e+00,0.895000,127.374302,1.119000,0.008785
6,2022,LTU,ARE,491000,4.510000e+02,6.014190e+02,6.000000e+01,5.370000e+01,0.895000,7.516667,11.199609,1.489970
7,2017,CZE,HUN,480257,2.234937e+06,2.394711e+06,1.666030e+06,1.861484e+06,0.895001,1.341475,1.286453,0.958984
8,2022,SWE,USA,392490,2.968188e+06,2.614010e+06,1.850925e+05,2.068070e+05,0.895001,16.036243,12.639853,0.788205
9,2017,FRA,JPN,481830,2.995253e+04,3.621705e+04,4.100000e+03,4.581000e+03,0.895001,7.305496,7.905928,1.082189


## Modified Direct Factor Example 

In [8]:
import duckdb

prepared_path = (
    "/storage/vacoda/data/interim/cif_fob/target_h5/prepared/"
    "comtrade_target_h5_cif_fob_prepared.parquet"
)

valid_direct_examples = duckdb.sql(
    f"""
    WITH import_rows AS (
        SELECT
            year,
            partner_iso AS exporter_iso,
            reporter_iso AS importer_iso,
            target_cmd,

            primary_value_hs_synced,

            CASE
                WHEN cif_value_usd > 0
                 AND fob_value_usd > 0
                THEN cif_value_usd * allocation_weight
            END AS paired_cif_value,

            CASE
                WHEN cif_value_usd > 0
                 AND fob_value_usd > 0
                THEN fob_value_usd * allocation_weight
            END AS paired_fob_value,

            CASE
                WHEN cif_value_usd > 0
                 AND fob_value_usd > 0
                THEN 1
                ELSE 0
            END AS has_paired_values

        FROM read_parquet('{prepared_path}')

        WHERE flow_code = 'M'
    ),

    aggregated AS (
        SELECT
            year,
            exporter_iso,
            importer_iso,
            target_cmd,

            COUNT(*) AS import_source_rows,

            SUM(has_paired_values)
                AS paired_source_rows,

            SUM(primary_value_hs_synced)
                AS total_import_value_cif,

            SUM(paired_cif_value)
                AS paired_cif_value,

            SUM(paired_fob_value)
                AS paired_fob_value

        FROM import_rows

        GROUP BY
            year,
            exporter_iso,
            importer_iso,
            target_cmd
    ),

    calculated AS (
        SELECT
            *,

            paired_cif_value / paired_fob_value
                AS direct_cif_fob_factor,

            paired_cif_value / total_import_value_cif
                AS direct_factor_value_coverage,

            paired_source_rows::DOUBLE
                / import_source_rows
                AS direct_factor_row_coverage

        FROM aggregated

        WHERE paired_cif_value > 0
          AND paired_fob_value > 0
          AND total_import_value_cif > 0
    )

    SELECT *
    FROM calculated

    WHERE direct_factor_value_coverage >= 0.90
      AND direct_cif_fob_factor BETWEEN 1.0 AND 1.5

    ORDER BY RANDOM()
    LIMIT 20
    """
).df()

valid_direct_examples

,year,exporter_iso,importer_iso,target_cmd,import_source_rows,paired_source_rows,total_import_value_cif,paired_cif_value,paired_fob_value,direct_cif_fob_factor,direct_factor_value_coverage,direct_factor_row_coverage
0,2017,ITA,PER,481031,4,4.0,3.784629e+05,3.784629e+05,3.635356e+05,1.041062,1.0,1.0
1,2021,CYP,SRB,940390,8,8.0,9.600000e+01,9.600000e+01,9.600000e+01,1.000000,1.0,1.0
2,2024,TUR,ROU,480592,4,4.0,1.070160e+05,1.070160e+05,1.027025e+05,1.042000,1.0,1.0
3,2024,CAN,PER,480519,4,4.0,4.865206e+05,4.865206e+05,3.957013e+05,1.229515,1.0,1.0
4,2017,SLV,HND,481149,2,2.0,1.233130e+04,1.233130e+04,1.147006e+04,1.075086,1.0,1.0
5,2021,ITA,PHL,551613,1,1.0,7.313000e+03,7.313000e+03,7.122000e+03,1.026818,1.0,1.0
6,2018,MKD,SRB,470790,8,8.0,8.098256e+06,8.098256e+06,8.049672e+06,1.006036,1.0,1.0
7,2021,W00,ECU,540500,8,8.0,2.799592e+04,2.799592e+04,2.521440e+04,1.110315,1.0,1.0
8,2019,AUT,COL,480100,12,12.0,2.334339e+05,2.334339e+05,2.089349e+05,1.117257,1.0,1.0
9,2022,DEU,CHL,481032,1,1.0,3.820530e+04,3.820530e+04,3.404106e+04,1.122330,1.0,1.0


In [9]:
example = valid_direct_examples.iloc[0]

source_rows = duckdb.sql(
    f"""
    SELECT *
    FROM read_parquet('{prepared_path}')
    WHERE flow_code = 'M'
      AND year = {int(example["year"])}
      AND reporter_iso = '{example["importer_iso"]}'
      AND partner_iso = '{example["exporter_iso"]}'
      AND target_cmd = '{example["target_cmd"]}'
    """
).df()

source_rows

,hs_sync_input_row_id,type_code,frequency_code,ref_period_id,year,month,period,reporter_code,reporter_iso,reporter_name,...,hs_sync_method,allocation_strategy,allocation_is_assumption,primary_value_original,primary_value_hs_synced,canonical_weight_kg,canonical_weight_source,canonical_weight_estimated,canonical_weight_is_proxy,canonical_weight_hs_synced
0,21455428,C,A,20170101,2017,52,2017,604,PER,Peru,...,already_target,deterministic,False,94615.73,94615.73,50630.0,reported_net_weight_kg,False,False,50630.0
1,21455431,C,A,20170101,2017,52,2017,604,PER,Peru,...,already_target,deterministic,False,94615.73,94615.73,50630.0,reported_net_weight_kg,False,False,50630.0
2,21455432,C,A,20170101,2017,52,2017,604,PER,Peru,...,already_target,deterministic,False,94615.73,94615.73,50630.0,reported_net_weight_kg,False,False,50630.0
3,21455567,C,A,20170101,2017,52,2017,604,PER,Peru,...,already_target,deterministic,False,94615.73,94615.73,50630.0,reported_net_weight_kg,False,False,50630.0


In [10]:
varying_columns = [
    column
    for column in source_rows.columns
    if source_rows[column].nunique(dropna=False) > 1
]

varying_columns


['hs_sync_input_row_id',
 'partner2_code',
 'partner2_iso',
 'partner2_name',
 'mot_code',
 'mot_name',
 'is_reported',
 'is_aggregate']

In [11]:
columns_to_show = [
    "year",
    "reporter_iso",
    "partner_iso",
    "flow_code",
    "target_cmd",
    "cif_value_usd",
    "fob_value_usd",
    "primary_value_hs_synced",
    "allocation_weight",
] + [
    column
    for column in varying_columns
    if column not in {
        "year",
        "reporter_iso",
        "partner_iso",
        "flow_code",
        "target_cmd",
        "cif_value_usd",
        "fob_value_usd",
        "primary_value_hs_synced",
        "allocation_weight",
    }
]

source_rows[columns_to_show]

,year,reporter_iso,partner_iso,flow_code,target_cmd,cif_value_usd,fob_value_usd,primary_value_hs_synced,allocation_weight,hs_sync_input_row_id,partner2_code,partner2_iso,partner2_name,mot_code,mot_name,is_reported,is_aggregate
0,2017,PER,ITA,M,481031,94615.73,90883.9,94615.73,1.0,21455428,380,ITA,Italy,2100,Sea,True,False
1,2017,PER,ITA,M,481031,94615.73,90883.9,94615.73,1.0,21455431,0,W00,World,2100,Sea,False,True
2,2017,PER,ITA,M,481031,94615.73,90883.9,94615.73,1.0,21455432,0,W00,World,0,TOTAL MOT,False,True
3,2017,PER,ITA,M,481031,94615.73,90883.9,94615.73,1.0,21455567,380,ITA,Italy,0,TOTAL MOT,False,True


In [13]:
check = duckdb.sql(
    f"""
    SELECT
        year,
        reporter_iso,
        partner_iso,
        flow_code,
        target_cmd,
        partner2_iso,
        mot_code,
        mot_name,
        is_reported,
        is_aggregate,
        cif_value_usd,
        fob_value_usd
    FROM read_parquet('{prepared_path}')
    WHERE year = 2017
      AND reporter_iso = 'PER'
      AND partner_iso = 'ITA'
      AND target_cmd = 481031
    ORDER BY
        mot_code,
        partner2_iso
    """
).df()

check

,year,reporter_iso,partner_iso,flow_code,target_cmd,partner2_iso,mot_code,mot_name,is_reported,is_aggregate,cif_value_usd,fob_value_usd
0,2017,PER,ITA,M,481031,ITA,0,TOTAL MOT,False,True,94615.73,90883.9
1,2017,PER,ITA,M,481031,W00,0,TOTAL MOT,False,True,94615.73,90883.9
2,2017,PER,ITA,M,481031,ITA,2100,Sea,True,False,94615.73,90883.9
3,2017,PER,ITA,M,481031,W00,2100,Sea,False,True,94615.73,90883.9
